# League Of Legends 15 Minute Match Predictor

**Name(s)**: Don Le

**Website Link**: [League Of Legends Match Predictor Website](https://donle727.github.io/League-Of-legends-Pro-League-Match-Predictor/)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px
pd.options.plotting.backend = 'plotly'

from dsc80_utils import * # Feel free to uncomment and use this.

## Introduction

For this project, we are analyizing competitive League of Legends esports sourced from [Oracle's Elixer data](https://oracleselixir.com/tools/downloads). This dataset provides post-match statistics from official professional leagues from 2014 to 2026. 

Despite the many professional leagues that are provided with the data, we will focus on the high-skill and high-staked, namely, Tier-1 leagues (CBLOL, LCK, LCP, LCS, LEC, and LPL). By narrowing our scope to these intense and unforgiving environments, we can better understand optimal gameplay and precise strategy with minimal noise often found in lower-tier professional play.

During our initial exploration of the data, we came up with some intersting questions regarding what actually drives a professional team to victory:
* "Is there any difference in kills between professional leagues?"
    * What's the difference in intensity and aggression between Tier-1 leagues and the rest of the competitive pool?
* "How can first objective (like dragon) can affect winrate?"
    * Does priority of objectives allow a team to leverage victory against their opponent? 
* "How much does early-game snowballing effect win rate?"
    * How much does early-game gold advantage help a team to win?

## Data Cleaning and Exploratory Data Analysis

### DataFrame

In [ ]:
# DataFrame Initialization (Raw Data)

csv_files = ['2023_LoL_esports_match_data_from_OraclesElixir.csv',
             '2024_LoL_esports_match_data_from_OraclesElixir.csv', 
             '2025_LoL_esports_match_data_from_OraclesElixir.csv']

data_folder = Path('data')

main_df = pd.concat([pd.read_csv(data_folder / f, low_memory=False) for f in csv_files], ignore_index=True) # Concatenate LoL Pro games from 2023-2026

main_df

,gameid,datacompleteness,url,league,...,deathsat25,opp_killsat25,opp_assistsat25,opp_deathsat25
0,ESPORTSTMNT06_2753012,complete,NaN,LFL2,...,0.0,0.0,1.0,0.0
1,ESPORTSTMNT06_2753012,complete,NaN,LFL2,...,1.0,0.0,1.0,0.0
2,ESPORTSTMNT06_2753012,complete,NaN,LFL2,...,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
376449,LOLTMNT03_332179,complete,NaN,DCup,...,1.0,1.0,1.0,4.0
376450,LOLTMNT03_332179,complete,NaN,DCup,...,13.0,13.0,34.0,8.0
376451,LOLTMNT03_332179,complete,NaN,DCup,...,8.0,8.0,18.0,13.0


### Leagues

In [ ]:
print(list(main_df['league'].unique()))

['LFL2', 'DDH', 'EL', 'LPL', 'GL', 'LCKC', 'NEXO', 'UL', 'LVP SL', 'LCK', 'LFL', 'PRM', 'LMF', 'SL (LATAM)', 'VL', 'CBLOL', 'LEC', 'NACL', 'LCO', 'CBLOLA', 'LHE', 'NLC', 'GLL', 'ESLOL', 'LLA', 'EBL', 'TCL', 'PGN', 'LPLOL', 'LCS', 'HM', 'LJL', 'HC', 'AL', 'PCS', 'LDL', 'VCS', 'EM', 'MSI', 'LAS', 'LRN', 'LRS', 'EPL', 'LJLA', 'CT', 'WLDs', 'PCL', 'ASCI', 'CDF', 'IC', 'USP', 'DCup', 'TSC', 'LIT', 'IGNIS', 'EWC', 'AC', 'PRMP', 'HW', 'NLC Aurora Open', 'EBLPA', 'GLLPA', 'KeSPA', 'LCP', 'HLL', 'LTA S', 'LTA N', 'RL', 'CD', 'ROL', 'LTA', 'FST', 'Asia Master', 'ASI']


### Columns

In [ ]:
print(list(main_df.columns))

['gameid', 'datacompleteness', 'url', 'league', 'year', 'split', 'playoffs', 'date', 'game', 'patch', 'participantid', 'side', 'position', 'playername', 'playerid', 'teamname', 'teamid', 'firstPick', 'champion', 'ban1', 'ban2', 'ban3', 'ban4', 'ban5', 'pick1', 'pick2', 'pick3', 'pick4', 'pick5', 'gamelength', 'result', 'kills', 'deaths', 'assists', 'teamkills', 'teamdeaths', 'doublekills', 'triplekills', 'quadrakills', 'pentakills', 'firstblood', 'firstbloodkill', 'firstbloodassist', 'firstbloodvictim', 'team kpm', 'ckpm', 'firstdragon', 'dragons', 'opp_dragons', 'elementaldrakes', 'opp_elementaldrakes', 'infernals', 'mountains', 'clouds', 'oceans', 'chemtechs', 'hextechs', 'dragons (type unknown)', 'elders', 'opp_elders', 'firstherald', 'heralds', 'opp_heralds', 'void_grubs', 'opp_void_grubs', 'firstbaron', 'barons', 'opp_barons', 'atakhans', 'opp_atakhans', 'firsttower', 'towers', 'opp_towers', 'firstmidtower', 'firsttothreetowers', 'turretplates', 'opp_turretplates', 'inhibitors', '

### Any Missing Values?

In [ ]:
main_df['firstdragon'].isna().sum()

316830

### Clean

In [ ]:
keep_cols = ['league', 'side', 'position', 'result', 'firstblood', 'firstdragon', 'golddiffat15', 'killsat15', 'ckpm']
tier_one_leagues = ['LCK', 'LEC', 'LCS', 'CBLOL', 'LCP', 'LPL']

df_clean = (
    main_df[keep_cols]
    .loc[main_df['league'].isin(tier_one_leagues)]
    .loc[main_df['position'] == 'team']
    .assign(
        result=lambda x: x['result'].astype(int),
        firstblood=lambda x: x['firstblood'].astype('Int64'),
        firstdragon=lambda x: x['firstdragon'].astype('Int64')
    )
    .reset_index(drop=True)
)

display(df_clean.head())

,league,side,position,result,...,firstdragon,golddiffat15,killsat15,ckpm
0,LPL,Blue,team,0,...,0,NaN,NaN,0.72
1,LPL,Red,team,1,...,1,NaN,NaN,0.72
2,LPL,Blue,team,0,...,1,NaN,NaN,0.71
3,LPL,Red,team,1,...,0,NaN,NaN,0.71
4,LPL,Blue,team,0,...,1,NaN,NaN,0.69


In [ ]:
agg_stats = (
    df_clean.groupby('league')
    .agg(
        avg_ckpm=('ckpm', 'mean'),
        avg_kills_at_15=('killsat15', 'mean'),
        first_blood_rate=('firstblood', 'mean'), # Since it's 1/0, the mean is the percentage!
        first_dragon=('firstdragon','mean')
    )
    .round(3)
    .sort_values(by='avg_ckpm', ascending=False)
)

display(agg_stats)

,avg_ckpm,avg_kills_at_15,first_blood_rate,first_dragon
league,,,,
LCP,0.87,3.41,0.5,0.5
LPL,0.86,NaN,0.5,0.5
LEC,0.85,3.76,0.5,0.5
CBLOL,0.83,2.72,0.5,0.5
LCS,0.81,3.34,0.5,0.5
LCK,0.80,3.01,0.5,0.5


In [ ]:
# Plotting the distribution of Gold Difference at 15 minutes
fig_uni = px.histogram(
    df_clean, 
    x="golddiffat15", 
    nbins=50,
    title="Distribution of Gold Difference at 15 Minutes (Tier-1 Leagues)",
    labels={"golddiffat15": "Gold Difference", "count": "Number of Games"},
    color_discrete_sequence=['#636EFA']
)

# Add a vertical line at 0 to show the perfect center
fig_uni.add_vline(x=0, line_dash="dash", line_color="black", annotation_text="Dead Even (0g)")

fig_uni.update_layout(width=800, height=500)
fig_uni.show()
fig_uni.write_html('assets/dist_of_gold_15')


### Bivariate Analysis #1

In [ ]:
# Get main_df with all other leagues
df_eda = main_df[main_df['position'] == 'team'].copy()


df_eda['league_group'] = df_eda['league'].apply(
    lambda x: 'Tier One' if x in tier_one_leagues else 'Other Leagues'
)

# Group by league
league_order = df_eda.groupby("league")['ckpm'].mean().sort_values(ascending=False).index

fig = px.box(
    df_eda, 
    x="league",
    y="ckpm",
    color="league_group",
    category_orders={'league': league_order},
    color_discrete_map={'Tier One': '#636EFA', 'Other Leagues': '#EF553B'},
    title="Distribution of CKPM by League",
    points="outliers",
)

fig.update_layout(
    width=1200, 
    height=600,
    title_x=0.5 
)

fig.show()

## Assessment of Missingness

Upon further examination of our dataset, LPL unfortunately did not measure data on kills and gold difference at 15 minute. Therefore, our research will have to exclude LPL entirely. We assume this to be Missing by Design because of how Chinese matches are hosted differently from those hosted by Riot themselves (foot note 1). There also other some observations that were missing from specific variables such as 'firstdragon'.


Foot Note:
1. [Source](https://github.com/FloPrm/lol_analytics?tab=readme-ov-file#tv-esports-data)

In [ ]:
missing_fifteen_stats = main_df[main_df['league'] == 'LPL'][['killsat15','golddiffat15']]
missing_fifteen_stats
print(f'Missing Observations: {missing_fifteen_stats['golddiffat15'].isna().sum()}')
print(f'LPL League Games: {missing_fifteen_stats.shape[0]}')


Missing Observations: 27324
LPL League Games: 27324


In [ ]:
# 1. See total missing values for firstdragon
print(f"Total missing firstdragon: {main_df['firstdragon'].isna().sum()}")

# 2. See which leagues have the most missing firstdragon data
missing_dragons = main_df[main_df['firstdragon'].isna()]
print(missing_dragons['league'].value_counts())

Total missing firstdragon: 316830
league
LPL      22776
LDL      17052
NACL     16310
         ...  
AC         350
FST        350
EBLPA      250
Name: count, Length: 74, dtype: int64


In [ ]:

target_col = 'firstdragon'
dependency_col = 'league'
# dependency_col = 'side'    # Swap to this for your MCAR test (Test 2)


missing_df = main_df[[target_col, dependency_col]].copy()


missing_df['is_missing'] = missing_df[target_col].isna()


def calculate_tvd(df, missing_col, dep_col):

    dist = df.pivot_table(
        index=dep_col, 
        columns=missing_col, 
        aggfunc='size', 
        fill_value=0
    )

    dist = dist / dist.sum() 
    
    tvd = dist.diff(axis=1).iloc[:, -1].abs().sum() / 2
    return tvd

observed_tvd = calculate_tvd(missing_df, 'is_missing', dependency_col)
print(f"Observed TVD: {observed_tvd:.4f}")

# Permutation Test
n_repetitions = 500 
tvds = []

for _ in range(n_repetitions):
    shuffled_col = np.random.permutation(missing_df[dependency_col])
    
    temp_df = missing_df.assign(shuffled=shuffled_col)
    simulated_tvd = calculate_tvd(temp_df, 'is_missing', 'shuffled')
    tvds.append(simulated_tvd)

# P-value
p_value = np.count_nonzero(np.array(tvds) >= observed_tvd) / n_repetitions
print(f"P-Value: {p_value:.4f}")

Observed TVD: 0.0588
P-Value: 0.0000


## Hypothesis Testing

1. Null Hypothesis: The Mean Combined Kills per Minute (CKPM) in Tier 1 Leagues is the same as the mean CKPM of other Leagues.
2. Alternative Hypothesis: The CKPM of Tier 1 Leagues is ***less*** than the mean CKPM of other leagues. 

In [ ]:
# 1. We assume 'df' is your full combined dataset from Step 1
# Get game-level data for ALL leagues
hyp_df = main_df[main_df['position'] == 'team'].drop_duplicates('gameid').copy()

# 2. Define Tier-One and create a True/False column
tier_one = ['CBLOL','LCK','LCP','LSC','LEC','LPL']
hyp_df['is_tier_one'] = hyp_df['league'].isin(tier_one)

# 3. Split into the two groups
tier_1_action = hyp_df[hyp_df['is_tier_one'] == True]['ckpm']
other_action = hyp_df[hyp_df['is_tier_one'] == False]['ckpm']

# 4. Calculate the Observed Difference (Tier 1 - Others)
# We expect this to be a NEGATIVE number if Tier 1 is less bloody
observed_diff = tier_1_action.mean() - other_action.mean()

print(f"Tier-One Mean CKPM: {tier_1_action.mean():.4f}")
print(f"Other Leagues Mean CKPM: {other_action.mean():.4f}")
print(f"Observed Test Statistic (Difference in Means): {observed_diff:.4f}")

# 1. Setup the data arrays
ckpm_values = hyp_df['ckpm'].values
n_tier_one = hyp_df['is_tier_one'].sum() # Total number of Tier-One games

n_permutations = 10000
differences = []

# 2. The Permutation Loop
for _ in range(n_permutations):
    # Shuffle all the CKPM values
    shuffled_ckpm = np.random.permutation(ckpm_values)
    
    # Split into fake groups based on the original group sizes
    fake_tier_1 = shuffled_ckpm[:n_tier_one]
    fake_others = shuffled_ckpm[n_tier_one:]
    
    # Calculate the fake difference in means
    fake_diff = fake_tier_1.mean() - fake_others.mean()
    differences.append(fake_diff)

differences_array = np.array(differences)

# 3. Calculate the P-value
# We use <= because we are testing if Tier 1 is significantly LOWER
p_value = np.mean(differences_array <= observed_diff)

print(f"Observed Difference: {observed_diff:.4f}")
print(f"P-value: {p_value}")

# 4. Visualize the Results
dist_df = pd.DataFrame({'diffs': differences})

fig_perm = px.histogram(
    dist_df, 
    x='diffs', 
    nbins=50,
    title=f'Permutation Test: Tier-One vs Other Leagues<br><sup>P-value: {p_value}</sup>',
    labels={'diffs': 'Difference in Mean CKPM (Tier-One - Others)'},
    template='plotly_white'
)

# Add the red line for our actual observed difference
fig_perm.add_vline(
    x=observed_diff, 
    line_dash="dash", 
    line_color="red", 
    annotation_text=f"Observed: {observed_diff:.4f}", 
    annotation_position="top left" # Positioned left because it's a negative value
)

fig_perm.show()

Tier-One Mean CKPM: 0.8393
Other Leagues Mean CKPM: 1.0041
Observed Test Statistic (Difference in Means): -0.1648
Observed Difference: -0.1648
P-value: 0.0


## Framing a Prediction Problem

In [ ]:
# TODO

## Baseline Model (Logistic Regression)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# 1. Prepare Data (Only Gold this time)
model_leagues = ['LCK', 'LEC', 'LCS', 'CBLOL', 'LCP'] 
model_df = df_clean[df_clean['league'].isin(model_leagues)].dropna(subset=['golddiffat15', 'result'])

X = model_df[['golddiffat15']]
y = model_df['result']

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Pipeline
baseline_model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])

# 4. Train & Score
baseline_model.fit(X_train, y_train)

print(f"Train Accuracy: {baseline_model.score(X_train, y_train):.4f}")
print(f"Test Accuracy: {baseline_model.score(X_test, y_test):.4f}")

Train Accuracy: 0.7155
Test Accuracy: 0.7224


In [ ]:


# 1. Create a range of gold differences from -5000 to +5000
gold_range = np.linspace(-5000, 5000, 500).reshape(-1, 1)

# 2. Provide a dummy 'killsat15' value (e.g., 0)
dummy_kills = np.zeros_like(gold_range)
X_plot_array = np.hstack([gold_range, dummy_kills])

# FIX 1: Convert to DataFrame so scikit-learn sees the feature names
X_plot = pd.DataFrame(X_plot_array, columns=['golddiffat15', 'killsat15'])

# 3. Get the model's predicted probabilities for winning
probs = baseline_model.predict_proba(X_plot)[:, 1]

# 4. Plot the Logistic S-Curve
fig = go.Figure()

fig.add_trace(go.Scatter(x=gold_range.flatten(), y=probs, name='Win Probability', line=dict(color='blue')))

# FIX 2: Draw the red threshold line manually so it never fails to render
fig.add_trace(go.Scatter(
    x=[-5000, 5000], 
    y=[0.5, 0.5], 
    mode="lines", 
    line=dict(color="red", dash="dash"), 
    name="50/50 Threshold"
))

fig.update_layout(
    title="Model Win Probability vs. Gold Differential at 15m",
    xaxis_title="Gold Difference at 15 Minutes",
    yaxis_title="Probability of Winning",
    width=900, height=500
)

fig.show()

In [ ]:


# 1. Get predictions on the test set
y_pred = baseline_model.predict(X_test)

# 2. Calculate the matrix
cm = confusion_matrix(y_test, y_pred)

# 3. Plot it as a heatmap
fig_cm = px.imshow(
    cm, 
    text_auto=True, 
    aspect="auto",
    labels=dict(x="Predicted Result", y="Actual Result"),
    x=['Loss (0)', 'Win (1)'],
    y=['Loss (0)', 'Win (1)'],
    title="Baseline Model: Confusion Matrix",
    color_continuous_scale='Blues'
)

fig_cm.show()

## Final Model (Random Forest)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

# 1. Clean data and split in just a few lines
features = ['golddiffat15', 'killsat15', 'firstblood', 'firstdragon']
df_f = df_clean[df_clean['league'].isin(['LCK', 'LEC', 'LCS'])].dropna(subset=features + ['result'])

X_train, X_test, y_train, y_test = train_test_split(df_f[features], df_f['result'], test_size=0.2, random_state=42)

# 2. Build Pipeline using the "remainder" shortcut
final_pl = Pipeline([
    ('prep', ColumnTransformer([('num', StandardScaler(), ['golddiffat15', 'killsat15'])], remainder='passthrough')),
    ('rf', RandomForestClassifier(max_depth=5, random_state=42))
])

# 3. Train and Score
final_pl.fit(X_train, y_train)
print(f"Final Model Test Accuracy: {final_pl.score(X_test, y_test):.4f}")

Final Model Test Accuracy: 0.7361


In [ ]:
# 1. Extract the actual Random Forest model from inside your pipeline
rf_model = final_pl.named_steps['rf']

# 2. Extract the importances (a list of percentages that add up to 1.0)
importances = rf_model.feature_importances_

# 3. Define the feature names in the exact order they went through the ColumnTransformer
# (StandardScaler did gold and kills first, passthrough did the other two last)
feature_names = ['golddiffat15', 'killsat15', 'firstblood', 'firstdragon']

# 4. Put them into a DataFrame and sort them
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=True)

# 5. Create a horizontal bar chart
fig_importance = px.bar(
    importance_df, 
    x='Importance', 
    y='Feature', 
    orientation='h', # Horizontal makes labels easier to read
    title='Random Forest Model: Feature Importances',
    labels={'Importance': 'Relative Importance (0 to 1.0)', 'Feature': ''},
    color='Importance',
    color_continuous_scale='Blues'
)

fig_importance.update_layout(width=900, height=400, showlegend=False)
fig_importance.show()

## Step 8: Fairness Analysis

In [ ]:
# TODO